# 🔬 מחברת 4: עיבוד שפה טבעית (NLP) ומידול נושאים
## מבוא למדעי הרוח הדיגיטליים | אוניברסיטת אריאל | תשפ"ו
### פרופ' שי גורדין
---
NLP (Natural Language Processing) = עיבוד שפה טבעית – ענף של AI שעוסק בהבנה וניתוח של שפה אנושית.

**נושאים:** (1) מהו NLP ומה הכלים העיקריים (2) זיהוי ישויות בשם (NER) לעברית (3) ניתוח רגש (Sentiment Analysis) (4) מידול נושאים – LDA (5) Embedding ומרחב וקטורי (6) יישומים במדעי הרוח

**🎯 הקשר למחקר:** ניתוח הגניזה הקהירית, קורפוסים של כתבות עיתון, זיהוי דמויות היסטוריות בטקסטים.

## חלק א: מהו NLP?

NLP = ממשק בין בלשנות, מדעי המחשב ובינה מלאכותית.

### רמות הניתוח הבלשני:

| רמה | שם | דוגמה |
|-----|-----|--------|
| **תו** | Character | א, ב, ג |
| **מורפמה** | Morpheme | כתב+תי, שמ+ים |
| **מילה** | Token | "ירושלים" |
| **משפט** | Sentence | "ירושלים היא בירת ישראל" |
| **שיח** | Discourse | פסקה, מסמך |

### מטלות NLP עיקריות:

- **Tokenization** – פיצול לטוקנים (מילים/תווים)
- **POS Tagging** – תיוג חלקי דיבור (שם עצם, פועל...)
- **NER** – זיהוי ישויות בשם (אנשים, מקומות, ארגונים)
- **Sentiment Analysis** – ניתוח רגש (חיובי/שלילי/ניטרלי)
- **Topic Modeling** – גילוי נושאים נסתרים בקורפוס

### NLP לעברית – אתגרים:

- כתיב חסר (ניקוד)
- מורפולוגיה עשירה (שורשים, בניינים)
- אותיות סופיות (מ/ם, נ/ן, פ/ף...)
- כיוון RTL

📖 לקריאה: Goldberg, Y. (2017). *Neural Network Methods for Natural Language Processing*. Morgan & Claypool.

---

## 🚀 לפני שמתחילים – מדריך הגדרה מהירה

### שלב 1: פתחו את המחברת ב-Google Colab
לחצו על הכפתור הכחול ״Open in Colab״ בדף הקורס.

### שלב 2: שמרו עותק אישי
`File → Save a copy in Drive`  
(חובה — אחרת השינויים שלכם **לא יישמרו**!)

### שלב 3: הריצו את התאים **לפי הסדר** — מלמעלה למטה
- **Shift + Enter** = מריץ תא נוכחי ועובר לבא אחריו  
- **Ctrl + Enter** = מריץ תא נוכחי ונשאר בו

### ⏱️ זמן משוער

| | זמן | תוכן |
|---|---|---|
| 🏫 **בכיתה** | ~30 דקות | התקנה + טוקניזציה + **NER** (זיהוי ישויות) |
| 🏠 **בבית** | ~45 דקות | **LDA** (מידול נושאים) + Word Embeddings + שמירה |

> 📖 **כיתה הפוכה** — לפני השיעור קראו:
> - **חלק א׳**: מהו NLP? (תא טקסט ראשון)
> - **חלק ד׳**: מהו Topic Modeling / LDA? (תא טקסט)
>
> **בכיתה** נתמקד ב-NER — זיהוי ישויות בשם, שהוא הכלי המרכזי לחילוץ מידע לערך הוויקיפדי שלכם.
> **בבית** תריצו את LDA ותבדקו אילו נושאים עולים בקורפוס שלכם.

### ❓ נתקעתם?
1. קראו את הודעת השגיאה  
2. שאלו את Gemini ב-[AI Studio](https://aistudio.google.com/) – העתיקו את השגיאה ושאלו
3. שאלו את המרצה / הקבוצה

---


In [ ]:
import warnings, sys

# Intercept at output stage to suppress jupyter_client kernel warnings
# (filterwarnings cannot reach the kernel IPC layer between cells)
_orig_showwarning = warnings.showwarning
def _showwarning(msg, cat, fname, lineno, file=None, line=None):
    fname_str = str(fname) if fname else ''
    msg_str   = str(msg)
    if 'jupyter_client' in fname_str or 'utcnow' in msg_str or 'datetime.datetime' in msg_str:
        return
    _orig_showwarning(msg, cat, fname, lineno, file, line)
warnings.showwarning = _showwarning

# התקנת ספריות NLP
!pip install spacy gensim pyLDAvis -q
# הורדת מודל עברית לspaCy (אם קיים)
# !python -m spacy download he_core_news_sm  # מודל עברית – בדוק זמינות
!pip install sentence-transformers scikit-learn python-bidi -q
print("✅ ספריות NLP הותקנו!")
print("   • spacy          - NLP pipeline")
print("   • gensim         - Topic Modeling (LDA)")
print("   • pyLDAvis       - ויזואליזציה של נושאים")
print("   • sentence-transformers - Embeddings")
print("   • python-bidi    - טקסט RTL בגרפים")

In [ ]:
# יבוא ספריות
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import re
import time
from collections import Counter
from IPython.display import display, HTML
import os
from bidi.algorithm import get_display

def rtl(s):
    return get_display(str(s))

# NLP
import gensim
from gensim import corpora, models
from gensim.models import LdaModel
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import NMF, LatentDirichletAllocation as LDA_sklearn
from sklearn.metrics.pairwise import cosine_similarity

matplotlib.rcParams['axes.unicode_minus'] = False
print("✅ ספריות יובאו!")
print(f"   Gensim {gensim.__version__}")

## חלק ב: טוקניזציה ותיוג מורפולוגי

### מהי טוקניזציה?

**Tokenization** = פיצול טקסט ליחידות משמעות (tokens).

לעברית, זה מורכב יותר מאנגלית:

- "לבית" → [ל+בית] = 2 מורפמות
- "שלנו" → [של+נו] = גוף שני רבים
- "בבית" → [ב+בית]

### spaCy לעברית:

```python
import spacy
nlp = spacy.load("he_core_news_sm")
doc = nlp("ירושלים היא בירת מדינת ישראל")
for token in doc:
    print(token.text, token.pos_, token.dep_)
```

### חלקי דיבור (POS Tags):

| תג | שם | דוגמה |
|----|-----|--------|
| NOUN | שם עצם | ירושלים, חפירה |
| VERB | פועל | מצא, חפר |
| ADJ | שם תואר | עתיק, דיגיטלי |
| PROPN | שם פרטי | דוד, חמאס |
| NUM | מספר | שלושה, 42 |

In [3]:
# ============================================================
# טוקניזציה בסיסית לעברית
# ============================================================

def tokenize_hebrew(text):
    """
    טוקניזציה בסיסית לעברית – פיצול למילים עם ניקוי.
    הערה: לניתוח מתקדם, מומלץ להשתמש ב-HebSpaCy או YAP.
    """
    # הסרת HTML ו-URLs
    text = re.sub(r'http\S+', ' ', text)
    text = re.sub(r'<[^>]+>', ' ', text)
    # שמירת אותיות עבריות ורווחים
    text = re.sub(r'[^\u05D0-\u05EA\s]', ' ', text)
    # פיצול ושמירת מילים ≥ 2 אותיות
    tokens = [w for w in text.split() if len(w) >= 2]
    return tokens


def pos_tag_simple(tokens):
    """
    תיוג POS פשוט לעברית (ללא מודל ML).
    מבוסס על רשימות ידועות.
    """
    # רשימות לדוגמה (מינימליות)
    proper_nouns = {
        'ירושלים', 'תל', 'אביב', 'חיפה', 'באר', 'שבע', 'ישראל',
        'יהודה', 'שומרון', 'גליל', 'נגב', 'סיני', 'ירדן',
        'מגידו', 'חצור', 'לכיש', 'באר', 'שובה', 'ערד', 'תמנע'
    }
    
    verbs_patterns = ['חפר', 'מצא', 'גילה', 'נבנה', 'נהרס', 'שלט', 'כבש', 'ניצח']
    
    tagged = []
    for token in tokens:
        if token in proper_nouns:
            tagged.append((token, 'PROPN'))
        elif any(token.startswith(v) or token.endswith(v) for v in verbs_patterns):
            tagged.append((token, 'VERB'))
        elif len(token) <= 3:
            tagged.append((token, 'FUNC'))  # מילת תפקיד
        else:
            tagged.append((token, 'NOUN'))  # ברירת מחדל
    
    return tagged


# ── דוגמה ──
example_texts = [
    "חפירות ארכיאולוגיות בירושלים חשפו שכבות מהתקופה הברונזה",
    "תל מגידו מציג שכבות מהמאה העשירית לפנה ספירה",
    "הגניזה הקהירית מכילה מסמכים מהמאה התשיעית עד המאה העשרים"
]

print("🔍 דוגמאות טוקניזציה:")
print("=" * 55)
for text in example_texts:
    tokens = tokenize_hebrew(text)
    tagged = pos_tag_simple(tokens)
    
    print(f"\n📝 טקסט: {text[:50]}...")
    print(f"   טוקנים: {tokens[:8]}")
    print(f"   תיוג: {[(t, pos) for t, pos in tagged[:5]]}")

🔍 דוגמאות טוקניזציה:

📝 טקסט: חפירות ארכיאולוגיות בירושלים חשפו שכבות מהתקופה הב...
   טוקנים: ['חפירות', 'ארכיאולוגיות', 'בירושלים', 'חשפו', 'שכבות', 'מהתקופה', 'הברונזה']
   תיוג: [('חפירות', 'NOUN'), ('ארכיאולוגיות', 'NOUN'), ('בירושלים', 'NOUN'), ('חשפו', 'NOUN'), ('שכבות', 'NOUN')]

📝 טקסט: תל מגידו מציג שכבות מהמאה העשירית לפנה ספירה...
   טוקנים: ['תל', 'מגידו', 'מציג', 'שכבות', 'מהמאה', 'העשירית', 'לפנה', 'ספירה']
   תיוג: [('תל', 'PROPN'), ('מגידו', 'PROPN'), ('מציג', 'NOUN'), ('שכבות', 'NOUN'), ('מהמאה', 'NOUN')]

📝 טקסט: הגניזה הקהירית מכילה מסמכים מהמאה התשיעית עד המאה ...
   טוקנים: ['הגניזה', 'הקהירית', 'מכילה', 'מסמכים', 'מהמאה', 'התשיעית', 'עד', 'המאה']
   תיוג: [('הגניזה', 'NOUN'), ('הקהירית', 'NOUN'), ('מכילה', 'NOUN'), ('מסמכים', 'NOUN'), ('מהמאה', 'NOUN')]


## חלק ג: זיהוי ישויות בשם (NER)

### מהו NER?

**Named Entity Recognition** = זיהוי ישויות בשם בטקסט.

### סוגי ישויות:

| סוג | תיאור | דוגמה |
|-----|--------|--------|
| **PER** | אדם | "דוד המלך", "רבי עקיבא" |
| **LOC** | מקום | "ירושלים", "כינרת" |
| **ORG** | ארגון | "האוניברסיטה העברית" |
| **DATE** | תאריך/תקופה | "המאה ה-10 לפנה"ס" |
| **EVENT** | אירוע | "חורבן הבית" |

### חשיבות NER לאסטורים:

- **ניתוח רשתות חברתיות** – מי מזכיר את מי?
- **ממפה מקומות** – בניית GeoHumani
- **ציר זמן** – מתי מתרחשים אירועים?
- **ביוגרפיה** – מי היה מה ומתי?

In [4]:
# ============================================================
# זיהוי ישויות בשם (NER) לעברית
# ============================================================

# רשימות ישויות ידועות (Gazetteer approach)
GAZETTEER = {
    'PER': {  # אנשים
        'דוד', 'שלמה', 'שאול', 'אברהם', 'משה', 'יהושע',
        'בן גוריון', 'הרצל', 'רמב"ם', 'רש"י', 'ביאליק',
        'רבי עקיבא', 'יוסף פלביוס', 'בר כוכבא'
    },
    'LOC': {  # מקומות
        'ירושלים', 'תל אביב', 'חיפה', 'באר שבע', 'יריחו',
        'מגידו', 'חצור', 'לכיש', 'ערד', 'תמנע', 'מצדה',
        'כינרת', 'ים המלח', 'ירדן', 'נגב', 'גליל', 'יהודה',
        'שומרון', 'עמק יזרעאל', 'שפלה', 'חרמון', 'כרמל'
    },
    'ORG': {  # ארגונים
        'האוניברסיטה העברית', 'רשות העתיקות', 'מוזיאון ישראל',
        'האקדמיה ללשון עברית', 'הכנסת', 'מדינת ישראל',
        'הסוכנות היהודית', 'הבונד', 'הפועל'
    },
    'PERIOD': {  # תקופות
        'הברונזה', 'הברזל', 'הביזנטית', 'הצלבנית', 'העות\'מאנית',
        'הכנענית', 'הפרסית', 'ההלניסטית', 'הרומית', 'המנדטורית',
        'ימי הביניים', 'העת החדשה', 'העת העתיקה'
    }
}


def find_entities(text, gazetteer=None):
    """
    זיהוי ישויות בשם בטקסט עברי.
    
    שיטה: Gazetteer-based (מבוסס רשימות ידועות).
    
    לזיהוי מתקדם יותר: spaCy עם מודל עברית, YAP, HebSpaCy.
    
    פרמטרים:
        text      : הטקסט לניתוח
        gazetteer : מילון ישויות {סוג: set(ישויות)}
    
    מחזיר: רשימת (ישות, סוג, מיקום)
    """
    if gazetteer is None:
        gazetteer = GAZETTEER
    
    entities = []
    
    for entity_type, entity_list in gazetteer.items():
        for entity in entity_list:
            # חיפוש כל הופעות
            for match in re.finditer(re.escape(entity), text):
                entities.append({
                    'ישות':    entity,
                    'סוג':     entity_type,
                    'התחלה':   match.start(),
                    'סיום':    match.end()
                })
    
    # מיון לפי מיקום
    entities.sort(key=lambda x: x['התחלה'])
    return entities


def highlight_entities(text, entities):
    """
    הדגשת ישויות בטקסט (פורמט HTML).
    """
    COLORS = {
        'PER':    '#FFD700',   # זהב – אנשים
        'LOC':    '#90EE90',   # ירוק – מקומות
        'ORG':    '#87CEEB',   # כחול – ארגונים
        'PERIOD': '#FFB347',   # כתום – תקופות
    }
    
    result = text
    # מהסוף להתחלה כדי לא לשבור אינדקסים
    for entity in sorted(entities, key=lambda x: x['התחלה'], reverse=True):
        color = COLORS.get(entity['סוג'], '#DDDDDD')
        start, end = entity['התחלה'], entity['סיום']
        original = text[start:end]
        replacement = f'<mark style="background-color:{color}">{original}<sup style="font-size:8px">[{entity["סוג"]}]</sup></mark>'
        result = result[:start] + replacement + result[end:]
    
    return result


# ── דוגמאות ──
demo_texts = [
    "דוד המלך בנה את ירושלים לבירת ממלכת ישראל בתקופת הברזל",
    "חפירות ברשות העתיקות בתל מגידו חשפו ממצאים מהתקופה הכנענית",
    "הרמב\"ם נולד בקורדובה ועבר לקהיר שם כתב את משנה תורה"
]

print("🏷️  זיהוי ישויות בשם (NER):")
print("=" * 65)

for text in demo_texts:
    entities = find_entities(text)
    print(f"\n📝 {text}")
    print(f"   ישויות שזוהו: {len(entities)}")
    for e in entities:
        print(f"   [{e['סוג']:6}] {e['ישות']}")
    
    # הצגה בHTML
    highlighted = highlight_entities(text, entities)
    print()

print("📌 מקרא:")
print("  🟨 PER = אדם  |  🟩 LOC = מקום  |  🔵 ORG = ארגון  |  🟧 PERIOD = תקופה")

🏷️  זיהוי ישויות בשם (NER):

📝 דוד המלך בנה את ירושלים לבירת ממלכת ישראל בתקופת הברזל
   ישויות שזוהו: 3
   [PER   ] דוד
   [LOC   ] ירושלים
   [PERIOD] הברזל


📝 חפירות ברשות העתיקות בתל מגידו חשפו ממצאים מהתקופה הכנענית
   ישויות שזוהו: 3
   [ORG   ] רשות העתיקות
   [LOC   ] מגידו
   [PERIOD] הכנענית


📝 הרמב"ם נולד בקורדובה ועבר לקהיר שם כתב את משנה תורה
   ישויות שזוהו: 1
   [PER   ] רמב"ם

📌 מקרא:
  🟨 PER = אדם  |  🟩 LOC = מקום  |  🔵 ORG = ארגון  |  🟧 PERIOD = תקופה


---

> 🏠 **חלק זה נועד לבית** — בכיתה נדלג על LDA ישירות.
> לאחר השיעור, הריצו את תאי LDA על הקורפוס מהנושא האישי שלכם.
> לחלופין, צפו בהדגמת הוידאו הקצרה בדף הקורס.

## חלק ד: מידול נושאים (Topic Modeling)

### מהו Topic Modeling?

**Topic Modeling** = גילוי נושאים נסתרים בקורפוס גדול, **ללא** פיקוח (Unsupervised Learning).

### האלגוריתם המרכזי: LDA

**LDA** (Latent Dirichlet Allocation) הנחה:

- כל **מסמך** = תערובת של נושאים
- כל **נושא** = התפלגות מעל מילים
- **מילים** = הנראות; **נושאים** = הנסתר

### דוגמה:

```
מסמך: "חפירה בתל מגידו חשפה חרסים מהתקופה הכנענית"
נושא 1 (ארכיאולוגיה): חפירה=0.15, תל=0.12, חרסים=0.11
נושא 2 (תקופות): כנענית=0.18, ברזל=0.14, ברונזה=0.13
```

### שימושים במדעי הרוח:

- גילוי **נושאים מרכזיים** בקורפוס עיתונות היסטורית
- מעקב אחר **שינויים בשיח** לאורך זמן
- זיהוי **אשכולות** של טקסטים דומים

📖 Blei, D.M., Ng, A.Y., Jordan, M.I. (2003). *Latent Dirichlet Allocation*. JMLR.

In [5]:
# ============================================================
# מצב LDA — האם להריץ מידול נושאים?
# ============================================================
# True  = דלגו על LDA (ברירת מחדל בכיתה — חוסך ~5 דקות)
# False = הריצו LDA (לשימוש בבית על הקורפוס האישי שלכם)

SKIP_LDA = True   # ← שנו ל-False בבית

if SKIP_LDA:
    print("🟡 LDA מושבת — בכיתה נתמקד ב-NER")
    print("   🏠 הריצו בבית עם SKIP_LDA = False על הקורפוס שלכם")
else:
    print("🟢 LDA יופעל — אימון המודל עשוי לקחת 2–5 דקות")


🟡 LDA מושבת — בכיתה נתמקד ב-NER
   🏠 הריצו בבית עם SKIP_LDA = False על הקורפוס שלכם


In [6]:
if not SKIP_LDA:
    # ============================================================
    # הכנת הקורפוס למידול נושאים
    # ============================================================
    
    # טקסטים לדוגמה (בפרויקט אמיתי – נשתמש בקורפוס מוויקיפדיה)
    SAMPLE_CORPUS = [
        # ארכיאולוגיה
        "חפירות בתל מגידו חשפו שכבות מהתקופה הכנענית והברונזה",
        "ממצאים ארכיאולוגיים ברמלה כוללים קרמיקה מהתקופה האסלאמית",
        "חפירה בירושלים גילתה חותם מלכותי מהתקופה הברזל",
        "אתר ארכיאולוגי בתל חצור מציג שרידי חומה מהמאה העשירית",
        "ממצאי ברונזה מאתר לכיש מעידים על תרבות עשירה",
        "חפירות בתל ערד חשפו מקדש ישראלי מהתקופה המלוכה",
        "קרמיקה פלשתית נמצאה באתרים רבים בשפלה יהודה",
        
        # היסטוריה
        "דוד המלך ייסד את ירושלים לבירת ממלכת ישראל",
        "חורבן בית המקדש הראשון בשנת תקפ לפנה ספירה",
        "הגלות הבבלית השפיעה עמוקות על התרבות היהודית",
        "תקופת בית שני כוללת השפעה פרסית הלניסטית ורומית",
        "המרד הגדול ברומאים הסתיים בחורבן הבית השני",
        "תקופת הגאונים בבבל ייצרה ספרות הלכתית עשירה",
        "היגירה לספרד הוליכה לתרחיש של ימי הביניים",
        
        # מדעי הרוח הדיגיטליים
        "כלים דיגיטליים לניתוח טקסטים היסטוריים עבריים",
        "ויזואליזציה של נתונים ארכיאולוגיים באמצעות GIS",
        "מסדי נתונים לתיעוד ממצאים ארכיאולוגיים",
        "עיבוד שפה טבעית לניתוח הגניזה הקהירית",
        "מפות דיגיטליות לחקר ישובים היסטוריים",
        "בינה מלאכותית לזיהוי אתרים ארכיאולוגיים בצילומי אויר",
    ]
    
    # מילות עצירה עבריות
    STOPWORDS = set([
        'של', 'עם', 'את', 'אל', 'על', 'כי', 'כן', 'לא', 'הוא', 'היא',
        'הם', 'הן', 'זה', 'זו', 'כל', 'יש', 'אין', 'רק', 'גם',
        'אבל', 'אם', 'כאשר', 'בין', 'מהתקופה', 'מהמאה', 'לפנה',
        'ב', 'ו', 'ה', 'ל', 'מ', 'כ', 'ש'
    ])
    
    
    def preprocess_for_lda(texts, stopwords=None):
        """
        עיבוד טקסטים למידול LDA.
        
        שלבים: ניקוי → טוקניזציה → הסרת stop words → פילטור קצרות
        """
        if stopwords is None:
            stopwords = STOPWORDS
        
        processed = []
        for text in texts:
            # ניקוי
            text = re.sub(r'[^\u05D0-\u05EA\s]', ' ', text)
            # טוקניזציה וסינון
            tokens = [
                w for w in text.split()
                if len(w) >= 3 and w not in stopwords
            ]
            processed.append(tokens)
        
        return processed
    
    
    # עיבוד הקורפוס
    processed_corpus = preprocess_for_lda(SAMPLE_CORPUS)
    
    # יצירת מילון ו-Bag of Words
    dictionary = corpora.Dictionary(processed_corpus)
    bow_corpus  = [dictionary.doc2bow(doc) for doc in processed_corpus]
    
    print(f"✅ הקורפוס עובד!")
    print(f"   מסמכים: {len(processed_corpus)}")
    print(f"   מילים במילון: {len(dictionary)}")
    print(f"\n🔍 דוגמה – מסמך ראשון:")
    print(f"   מילים: {processed_corpus[0]}")
    print(f"   BoW: {bow_corpus[0][:5]}")

In [7]:
if not SKIP_LDA:
    # ============================================================
    # אימון מודל LDA
    # ============================================================
    
    print("🏋️  מאמן מודל LDA...")
    
    # הפרמטר החשוב ביותר: num_topics
    # בפרויקט אמיתי – נרצה לבדוק ערכים שונים ולמדוד Coherence Score
    NUM_TOPICS = 4  # ניסיון ראשוני
    
    lda_model = LdaModel(
        corpus=bow_corpus,
        id2word=dictionary,
        num_topics=NUM_TOPICS,
        random_state=42,         # לשחזוריות
        update_every=1,
        chunksize=10,
        passes=20,               # כמה פעמים לעבור על הקורפוס
        alpha='auto',            # אלפא = מידת ערבוב הנושאים
        per_word_topics=True
    )
    
    print(f"✅ מודל LDA אומן!")
    print(f"   נושאים: {NUM_TOPICS}")
    print()
    
    # הצגת הנושאים
    print("📊 הנושאים שהתגלו:")
    print("=" * 60)
    
    for topic_id, words in lda_model.print_topics(num_topics=NUM_TOPICS, num_words=8):
        print(f"\n  נושא {topic_id + 1}:")
        # פירסור המילים
        word_weights = []
        for item in words.split(' + '):
            weight, word = item.split('*')
            word = word.replace('\"', '')
            word_weights.append((word, float(weight)))
        
        for word, weight in word_weights:
            bar = '█' * int(weight * 200)
            print(f"    {word:<15} {weight:.4f} |{bar}")

In [ ]:
if not SKIP_LDA:
    # ============================================================
    # ויזואליזציה של הנושאים
    # ============================================================

    # ── גרף: מילים מובילות לכל נושא ──
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()

    colors_list = ['#e74c3c', '#2980b9', '#27ae60', '#f39c12']

    for topic_id in range(NUM_TOPICS):
        topic = lda_model.show_topic(topic_id, topn=10)
        words, weights = zip(*topic)

        ax = axes[topic_id]
        ax.barh(range(len(words)), weights,
                color=colors_list[topic_id], alpha=0.8)
        ax.set_yticks(range(len(words)))
        ax.set_yticklabels([rtl(w) for w in words], fontsize=11)
        ax.invert_yaxis()
        ax.set_title(rtl(f'נושא {topic_id + 1}'), fontsize=13, fontweight='bold',
                     color=colors_list[topic_id])
        ax.set_xlabel(rtl('משקל'), fontsize=10)

    plt.suptitle(rtl('מידול נושאים – LDA') + '\n' + rtl('מילים מובילות לכל נושא'),
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('01_lda_topics.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("💾 הגרף נשמר: 01_lda_topics.png")

    # ── ויזואליזציה אינטראקטיבית עם pyLDAvis ──
    print("\n🌐 יוצר ויזואליזציה אינטראקטיבית...")
    try:
        vis_data = gensimvis.prepare(lda_model, bow_corpus, dictionary)
        pyLDAvis.save_html(vis_data, '02_lda_interactive.html')
        print("✅ ויזואליזציה נשמרה: 02_lda_interactive.html")
        print("   פתחו את הקובץ בדפדפן לחקירה אינטראקטיבית!")
    except Exception as e:
        print(f"⚠️ ויזואליזציה pyLDAvis: {e}")
        print("   ניתן להמשיך – הגרפים הסטטיים זמינים")

## חלק ה: Word Embeddings – מילים כוקטורים

### מהו Word Embedding?

**Word Embedding** = ייצוג מילים כוקטורים במרחב רב-ממדי.

### הרעיון המרכזי:

> "מילים המופיעות בהקשרים דומים = ייצוגים דומים במרחב הוקטורי"

### דוגמאות קלאסיות:

- מלך − גבר + אישה ≈ מלכה
- ירושלים − ישראל + צרפת ≈ פריז

### מודלים מפורסמים:

| מודל | מפתח | שנה | מאפיין |
|------|-------|-----|--------|
| **Word2Vec** | Google | 2013 | מהיר, פשוט |
| **GloVe** | Stanford | 2014 | מבוסס קו-אוקורנס גלובלי |
| **FastText** | Meta | 2016 | טוב למורפולוגיה (= עברית!) |
| **BERT** | Google | 2018 | הקשרי, שינה את ה-NLP |

### FastText לעברית:

FastText מיוחד לשפות מורפולוגיות כמו עברית כי הוא עובד ברמת ה**תת-מילה** (subword).

In [ ]:
if not SKIP_LDA:
    # ============================================================
    # הדגמת Word Embeddings (גרסה פשוטה)
    # ============================================================
    # בגרסה פשוטה – נשתמש ב-TF-IDF matrix כייצוג וקטורי בסיסי
    # לייצוגים מתקדמים: FastText, BERT-Hebrew, AlephBERT

    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity

    print("🔢 מחשב ייצוגים וקטוריים (TF-IDF)...")

    # ייצוג TF-IDF
    tfidf    = TfidfVectorizer(analyzer=lambda x: x)
    tfidf_matrix = tfidf.fit_transform(processed_corpus)
    sim_matrix   = cosine_similarity(tfidf_matrix)

    print(f"  ממדי המטריצה: {tfidf_matrix.shape}")
    print(f"  ({tfidf_matrix.shape[0]} מסמכים × {tfidf_matrix.shape[1]} מילים)")

    # ── קטגוריות ──
    CAT_NAMES    = ['ארכיאולוגיה', 'היסטוריה', 'מדעי הרוח הדיגיטליים']
    CAT_INDICES  = [list(range(0, 7)), list(range(7, 14)), list(range(14, 20))]
    CAT_COLORS   = ['#e74c3c', '#2980b9', '#27ae60']

    # ── גרף: שני לוחות זה לצד זה ──
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

    # -------- לוח שמאל: מטריצת דמיון 20×20 --------
    im1 = ax1.imshow(sim_matrix, cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)
    plt.colorbar(im1, ax=ax1, label=rtl('דמיון קוסינוס'))

    # קווי הפרדה בין קטגוריות
    for sep in [6.5, 13.5]:
        ax1.axhline(sep, color='white', linewidth=2)
        ax1.axvline(sep, color='white', linewidth=2)

    # תוויות ציר – מרכז כל קטגוריה
    tick_pos   = [3, 10, 17]
    tick_lbls  = [rtl(n) for n in CAT_NAMES]
    ax1.set_xticks(tick_pos)
    ax1.set_xticklabels(tick_lbls, fontsize=10)
    ax1.set_yticks(tick_pos)
    ax1.set_yticklabels(tick_lbls, fontsize=10)
    ax1.set_title(rtl('מטריצת דמיון בין כל המסמכים') + '\n' +
                  rtl('(צהוב = דמיון גבוה, לבן = נמוך)'),
                  fontsize=12, fontweight='bold')

    # -------- לוח ימין: דמיון ממוצע בין קטגוריות 3×3 --------
    n = len(CAT_NAMES)
    mean_sim = np.zeros((n, n))
    for i, idx_i in enumerate(CAT_INDICES):
        for j, idx_j in enumerate(CAT_INDICES):
            block = sim_matrix[np.ix_(idx_i, idx_j)]
            if i == j:
                mask = ~np.eye(len(idx_i), dtype=bool)
                mean_sim[i, j] = block[mask].mean()
            else:
                mean_sim[i, j] = block.mean()

    im2 = ax2.imshow(mean_sim, cmap='Blues', aspect='auto', vmin=0, vmax=1)
    plt.colorbar(im2, ax=ax2, label=rtl('דמיון ממוצע'))

    cat_lbls_rtl = [rtl(c) for c in CAT_NAMES]
    ax2.set_xticks(range(n))
    ax2.set_xticklabels(cat_lbls_rtl, fontsize=11)
    ax2.set_yticks(range(n))
    ax2.set_yticklabels(cat_lbls_rtl, fontsize=11)

    # ערכים מספריים בתוך כל תא
    for i in range(n):
        for j in range(n):
            val = mean_sim[i, j]
            ax2.text(j, i, f'{val:.2f}',
                     ha='center', va='center', fontsize=14, fontweight='bold',
                     color='white' if val > 0.4 else 'black')

    ax2.set_title(rtl('דמיון ממוצע בין קטגוריות') + '\n' +
                  rtl('(ערך גבוה = מסמכים דומים יותר)'),
                  fontsize=12, fontweight='bold')

    plt.suptitle(rtl('ייצוג וקטורי של מסמכים — TF-IDF + Cosine Similarity'),
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('03_document_similarity.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("💾 הגרף נשמר: 03_document_similarity.png")

    # ── דמיון בין מסמכים ──
    print("\n📏 דמיון בין מסמכים (Cosine Similarity):")
    print(f"  מסמך 1 ↔ מסמך 2 (ארכיאולוגיה): {sim_matrix[0][1]:.3f}")
    print(f"  מסמך 1 ↔ מסמך 8 (היסטוריה):   {sim_matrix[0][7]:.3f}")
    print(f"  מסמך 1 ↔ מסמך 15 (DH):         {sim_matrix[0][14]:.3f}")

## חלק ו: יישומים במדעי הרוח

### דוגמאות מחקריות:

**1. ניתוח הגניזה הקהירית**  
מאגר Friedberg Genizah Project – שימוש ב-NLP לניתוח 330,000 קטעים.

**2. ניתוח עיתונות היסטורית**  
עיתונים עבריים מ-1870 עד 1948 – topic modeling לזיהוי שינויי שיח.

**3. זיהוי דמויות היסטוריות**  
NER + co-occurrence analysis = מי הכיר את מי בתקופה התלמודית?

**4. ניתוח ספרותי**  
השוואת קורפוסים: שפת ביאליק לעומת עגנון – מה מייחד כל אחד?

### כלים מתקדמים לעברית:

- **AlephBERT**: מודל BERT מאומן על עברית (AI21 Labs)
- **HebSpaCy**: spaCy עם תמיכה בעברית
- **MILA**: מאגר NLP לעברית (Technion)
- **DictaBERT**: BERT עם מורפולוגיה עברית

In [10]:
# ============================================================
# ניתוח רגש (Sentiment Analysis) – גרסה פשוטה
# ============================================================

# מילוני רגש עבריים בסיסיים
POSITIVE_WORDS = {
    'טוב', 'מצוין', 'נפלא', 'יפה', 'חשוב', 'מרשים', 'ייחודי',
    'נדיר', 'מדהים', 'משמעותי', 'חיוני', 'ערך', 'עשיר',
    'מתקדם', 'חדשני', 'פורה', 'הצלחה', 'ניצחון', 'שגשוג',
    'שלום', 'ברית', 'קדוש', 'חכמה', 'גדול', 'כביר'
}

NEGATIVE_WORDS = {
    'רע', 'גרוע', 'נורא', 'חרב', 'הרוס', 'נפל', 'מסכן',
    'מלחמה', 'כיבוש', 'גלות', 'גירוש', 'רצח', 'הרג',
    'חורבן', 'כישלון', 'אסון', 'שבי', 'עצב', 'בכי',
    'עוני', 'רעב', 'מחלה', 'אויב', 'פגע', 'הגלה'
}


def analyze_sentiment(text, pos_words=None, neg_words=None):
    """
    ניתוח רגש בסיסי לטקסט עברי.
    
    מחזיר: dict עם ציון רגש ומילים שהתגלו
    """
    if pos_words is None:
        pos_words = POSITIVE_WORDS
    if neg_words is None:
        neg_words = NEGATIVE_WORDS
    
    # ניקוי וטוקניזציה
    words = re.sub(r'[^\u05D0-\u05EA\s]', ' ', text).split()
    
    found_positive = [w for w in words if w in pos_words]
    found_negative = [w for w in words if w in neg_words]
    
    pos_score = len(found_positive)
    neg_score = len(found_negative)
    
    if pos_score > neg_score:
        sentiment = 'חיובי 😊'
    elif neg_score > pos_score:
        sentiment = 'שלילי 😞'
    else:
        sentiment = 'ניטרלי 😐'
    
    return {
        'רגש':           sentiment,
        'ציון_חיובי':    pos_score,
        'ציון_שלילי':    neg_score,
        'מילים_חיוביות': found_positive,
        'מילים_שליליות': found_negative
    }


# ── דוגמאות ──
print("😊 ניתוח רגש לטקסטים היסטוריים:")
print("=" * 65)

sentiment_examples = [
    "ירושלים שגשגה בתקופת שלמה המלך, המלך הגדול שבנה את המקדש הנפלא",
    "חורבן הבית הוביל לגלות ורצח והרס נורא של עם ישראל",
    "הממצאים הארכיאולוגיים מספקים מידע חשוב על העבר ההיסטורי",
    "המלחמה והכיבוש הרומי גרמו לאסון ולמחלה ולעוני בארץ"
]

for text in sentiment_examples:
    result = analyze_sentiment(text)
    print(f"\n📝 {text[:55]}...")
    print(f"   רגש: {result['רגש']}")
    print(f"   חיובי ({result['ציון_חיובי']}): {result['מילים_חיוביות']}")
    print(f"   שלילי ({result['ציון_שלילי']}): {result['מילים_שליליות']}")

print("\n⚠️ הערה: ניתוח רגש בסיסי בלבד – לניתוח מדויק:")
print("   pip install transformers  # AlephBERT בעברית")

😊 ניתוח רגש לטקסטים היסטוריים:

📝 ירושלים שגשגה בתקופת שלמה המלך, המלך הגדול שבנה את המקד...
   רגש: ניטרלי 😐
   חיובי (0): []
   שלילי (0): []

📝 חורבן הבית הוביל לגלות ורצח והרס נורא של עם ישראל...
   רגש: שלילי 😞
   חיובי (0): []
   שלילי (2): ['חורבן', 'נורא']

📝 הממצאים הארכיאולוגיים מספקים מידע חשוב על העבר ההיסטורי...
   רגש: חיובי 😊
   חיובי (1): ['חשוב']
   שלילי (0): []

📝 המלחמה והכיבוש הרומי גרמו לאסון ולמחלה ולעוני בארץ...
   רגש: ניטרלי 😐
   חיובי (0): []
   שלילי (0): []

⚠️ הערה: ניתוח רגש בסיסי בלבד – לניתוח מדויק:
   pip install transformers  # AlephBERT בעברית


In [11]:
if not SKIP_LDA:
    # ============================================================
    # שמירת הנתונים
    # ============================================================

    import pandas as pd

    print("💾 שומר נתונים...")

    # שמירת הקורפוס המעובד
    corpus_data = []
    for i, (original, processed) in enumerate(zip(SAMPLE_CORPUS, processed_corpus)):
        topic_dist = lda_model.get_document_topics(bow_corpus[i])
        top_topic = max(topic_dist, key=lambda x: x[1])[0] if topic_dist else -1
        corpus_data.append({
            'מזהה':    i + 1,
            'טקסט':    original,
            'מילים':   ' '.join(processed),
            'נושא_ראשי': top_topic + 1
        })

    df_corpus = pd.DataFrame(corpus_data)
    df_corpus.to_csv('nlp_corpus.csv', index=False, encoding='utf-8-sig')
    print("  ✅ nlp_corpus.csv")

    # שמירת נושאי LDA
    topics_data = []
    for topic_id in range(NUM_TOPICS):
        for word, weight in lda_model.show_topic(topic_id, topn=10):
            topics_data.append({
                'נושא':  topic_id + 1,
                'מילה':  word,
                'משקל':  round(weight, 4)
            })
    pd.DataFrame(topics_data).to_csv('lda_topics.csv', index=False, encoding='utf-8-sig')
    print("  ✅ lda_topics.csv")

    print("\n🎉 מחברת 4 הושלמה!")
    print("\n📚 הכלים המתקדמים לעברית:")
    print("  → AlephBERT: https://huggingface.co/onlplab/alephbert-base")
    print("  → DictaBERT: https://huggingface.co/dicta-il")
    print("  → HebSpaCy:  https://github.com/explosion/spacy-models")

---

## חלק ז: מקרה בוחן — ניתוח הגניזה הקהירית

**הגניזה הקהירית** היא אחד מגדולי אוצרות הכתובים בעולם: כ-300,000 קטעי כתב יד שנשמרו בגנזה של בית הכנסת בן עזרא בפוסטאט (קהיר), המתוארכים בין המאות ה-9 וה-19. הם כוללים מסמכים בעברית, ארמית, יהודית-ערבית ועוד — מכתבים אישיים, חוזים מסחריים, פרגמנטים של תלמוד, פיוטים ועוד.

### מסד הנתונים: MiDRASH Automatic Transcriptions (Zenodo)

> **Gordin, S. et al. (2025).** *MiDRASH Automatic Transcriptions of the Cairo Geniza Fragments.* Zenodo. [https://doi.org/10.5281/zenodo.17734473](https://doi.org/10.5281/zenodo.17734473)

מסד הנתונים מכיל **תמלולים אוטומטיים** (HTR) של כל אוסף תמונות הגניזה שבספרייה הלאומית של ישראל, שנוצרו במסגרת פרויקט MiDRASH של המועצה האירופית למחקר (ERC). הקובץ המלא הוא 444 MB; בתרגיל זה נעבוד עם מדגם של 12 קטעים.

**הצינור שנפעיל:**

| שלב | כלי | מה נגלה |
|-----|-----|---------|
| א | טוקניזציה + NER | אנשים, מקומות וארגונים שנזכרו |
| ב | ניתוח רגש | האם הקטע חיובי, שלילי או ניטרלי? |
| ג | LDA | אילו 3 נושאים מרכזיים עולים בקורפוס? |
| ד | דמיון וקטורי | אילו קטעים דומים זה לזה תוכנית? |

In [ ]:
# ============================================================
# טעינת מדגם מהגניזה הקהירית
# ============================================================
# המסד המלא (444 MB) זמין ב:
#   https://doi.org/10.5281/zenodo.17734473
#
# להורדה ועבודה עם הקובץ המלא (הריצו בבית):
#
#   import requests, zipfile, io
#   URL = ("https://zenodo.org/records/17734473/files/"
#          "MiDRASH_Geniza_Transcriptions_0.8.txt.zip?download=1")
#   r    = requests.get(URL, stream=True)
#   data = b''.join(r.iter_content(1024 * 1024))
#   with zipfile.ZipFile(io.BytesIO(data)) as zf:
#       full_text = zf.read(zf.namelist()[0]).decode('utf-8')
#   fragments = [f.strip() for f in full_text.split('\n\n') if len(f.strip()) > 50]
#
# לצורך הדגמה — 12 קטעים נציגים:

GENIZAH_TEXTS = {
    'T-S 8J22.30 (מכתב אישי)':
        'לאחי היקר שלום וברכה מן השמים יסגא בכל עת ובכל שעה '
        'ידוע יהיה לך כי הגעתי לפוסטאט בשלום ומצאתי את הבית ריק '
        'הבעל הבית תובע ממני שכר הבית ואין בידי כסף '
        'שלח לי בדחיפות מה שיכול בידך ואל תאחר '
        'ושלום עלייך ועל כל בני ביתך',

    'T-S AS 153.196 (חוזה מסחרי)':
        'הסכם שותפות בין יצחק בן אברהם הסוחר ובין שמואל בן יוסף '
        'לסחור בבדים ותבלין מאלכסנדריה לעדן '
        'ההון הראשוני מאה דינר זהב מכל אחד מן השותפים '
        'הרווחים יחולקו בשווה והפסדים כמו כן '
        'נחתם בפוסטאט בחודש תשרי',

    'T-S Misc.28.56 (שאלה ותשובה הלכתית)':
        'שאלה לרבינו הגדול גאון בבל על ענין הגט '
        'האישה שקיבלה גט מבעלה ואחר כך בא ואמר שלא כוון לגרשה '
        'תשובה הגט כשר ואין דבריו נשמעים לאחר שנמסר בידה '
        'כך פסקנו על פי הלכה ואין לחלוק על פסק זה',

    'T-S K25.67 (פרגמנט תלמודי)':
        'דתניא רבי אומר כל המזכה את הרבים זכות הרבים תלויה בו '
        'אבל המחטיא את הרבים אין מספיקין בידו לעשות תשובה '
        'מפני שחטאו של יחיד בינו לבין קונו '
        'אבל חטא שהחטיא את הרבים גדול עוונו מנשוא',

    'T-S 16.17 (פיוט לשבת)':
        'לכה דודי לקראת כלה פני שבת נקבלה '
        'בואי בשלום עטרת בעלה גם בשמחה ובצהלה '
        'תוך אמוני עם סגולה בואי כלה בואי כלה '
        'שמרו שבת מחללה וקידשו היום הגדול הזה',

    'T-S Ar.30.239 (ספר רפואה)':
        'לרפואת הקדחת קח עלי הקינמון ועצי האלוורה '
        'ובשל אותם במים עד שיישאר חצי ושתה בכל בוקר לפני האכילה '
        'וכן לרפואת כאב הבטן קח זרעי הכמון ובשל עם שמן זית '
        'כך כתב אבן סינא בספרו הגדול ועל פיו נוהגים הרופאים',

    'T-S NS 246.29 (תפילה)':
        'אדון עולם אשר מלך בטרם כל יציר נברא '
        'לעת נעשה בחפצו כל אזי מלך שמו נקרא '
        'ואחרי ככלות הכל לבדו ימלוך נורא '
        'והוא היה והוא הווה והוא יהיה בתפארה',

    'T-S 12.196 (רשומות קהילה)':
        'פנקס הקהל של הקהילה היהודית בפוסטאט בעיר מצרים '
        'נרשמו שמות הנדבות לבית הכנסת הגדול בראש השנה '
        'רבי מאיר בן שלמה הנגיד נדב עשרה דינרים '
        'הנשיא אברהם בן אשר נדב חמישה עשר דינרים '
        'יתמי שמואל הסוחר קיבלו תמיכה מקופת הצדקה',

    'T-S Ar.51.92 (פילוסופיה)':
        'יסודות הדת הבורא יתברך שמו הוא הראשון והאחרון ואין כמוהו '
        'הנבואה אמת ומשה רבינו אדון הנביאים '
        'התורה הזאת לא תוחלף ולא תהיה תורה אחרת '
        'השגחתו על בני אדם לפי מעשיהם ודרכיהם',

    'T-S 10J5.14 (כתובה)':
        'ביום ראשון בשבת בחמישה ימים לחודש אדר '
        'בעיר פוסטאט הידועה על נהר הנילוס '
        'אמר לה החתן יצחק בן יעקב לכלה מרים בת שמואל '
        'הרי את מקודשת לי במוהר ובמתנות ובתנאים הכתובים '
        'המוהר מאה דינר זהב ובגדים ותכשיטים',

    'T-S 8.14 (אגרת מסחרית מהים)':
        'כתבתי אגרת זו מן הספינה בדרך לעדן '
        'הסחורה שלחתי עשרה שקים פלפל ושלושה שקי ציפורן '
        'הסוחרים שותפי הם אברהם ויוסף בני משה מאלכסנדריה '
        'הים סוער ואנחנו מתפללים לשלום וכל הרווחים יחולקו',

    'T-S H15.47 (פרשנות מקרא)':
        'פירוש על פרשת בראשית בראשית ברא אלהים '
        'בשביל התורה שנקראת ראשית דרכו ובשביל ישראל שנקראו ראשית '
        'ואם תאמר לאומות העולם אין אתם גזלנים שכבשתם ארצות '
        'הוא ברא אותה ונתנה לאשר ישר בעיניו ברצונו נתנה להם',
}

print(f'✅ נטענו {len(GENIZAH_TEXTS)} קטעי גניזה')
for name, text in GENIZAH_TEXTS.items():
    print(f'  📜 {name:<42}  ({len(text.split())} מילים)')

In [ ]:
# ============================================================
# גניזה שלב א: טוקניזציה ו-NER
# ============================================================

# הרחבת מאגר הישויות לתקופת הגניזה
GENIZAH_GAZETTEER = {k: set(v) for k, v in GAZETTEER.items()}
GENIZAH_GAZETTEER['PER'] |= {
    'אברהם', 'יצחק', 'יעקב', 'שמואל', 'יוסף', 'משה', 'מרים', 'מאיר',
    'שלמה', 'אשר', 'אדון', 'יעקב',
}
GENIZAH_GAZETTEER['LOC'] |= {
    'פוסטאט', 'אלכסנדריה', 'עדן', 'בבל', 'קהיר', 'הנילוס', 'ספרד',
}
GENIZAH_GAZETTEER['ORG'] |= {
    'קופת הצדקה', 'בית הכנסת',
}

# הרצה על כל הקטעים
entity_counts   = Counter()
entities_by_doc = {}
for name, text in GENIZAH_TEXTS.items():
    ents = find_entities(text, GENIZAH_GAZETTEER)
    entities_by_doc[name] = ents
    for e in ents:
        entity_counts[e['סוג']] += 1

print('🏷️  ישויות שזוהו בגניזה:')
print('=' * 50)
for etype, count in entity_counts.most_common():
    print(f'  [{etype:6}] {count:3}  {"█" * count}')
print(f'\n  סה"כ: {sum(entity_counts.values())} ישויות ב-{len(GENIZAH_TEXTS)} קטעים')

# הדגמה עם הדגשת HTML
demo_key = 'T-S 12.196 (רשומות קהילה)'
html_out = highlight_entities(GENIZAH_TEXTS[demo_key], entities_by_doc[demo_key])
print(f'\n📜 הדגמת NER — {demo_key}:')
display(HTML(f'<div dir="rtl" style="font-size:15px;line-height:2.2">{html_out}</div>'))
print('📌 מקרא: 🟨 PER = אדם  |  🟩 LOC = מקום  |  🔵 ORG = ארגון  |  🟧 PERIOD = תקופה')

In [ ]:
# ============================================================
# גניזה שלב ב: ניתוח רגש
# ============================================================
sentiment_rows = []
for name, text in GENIZAH_TEXTS.items():
    res        = analyze_sentiment(text)
    short_name = name.split('(')[1].rstrip(')')
    sentiment_rows.append({
        'שם':    short_name,
        'חיובי': res['ציון_חיובי'],
        'שלילי': res['ציון_שלילי'],
        'נטייה': res['ציון_חיובי'] - res['ציון_שלילי'],
    })

df_sent = pd.DataFrame(sentiment_rows).sort_values('נטייה', ascending=True)

fig, ax = plt.subplots(figsize=(9, 7))
bar_colors = ['#c0392b' if v < 0 else '#27ae60' if v > 0 else '#95a5a6'
              for v in df_sent['נטייה']]
ax.barh(range(len(df_sent)), df_sent['נטייה'], color=bar_colors, alpha=0.85)
ax.set_yticks(range(len(df_sent)))
ax.set_yticklabels([rtl(n) for n in df_sent['שם']], fontsize=11)
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel(rtl('ניתוח רגש (חיובי − שלילי)'), fontsize=11)
ax.set_title(
    rtl('ניתוח רגש — קטעי הגניזה הקהירית') + '\n' +
    rtl('ירוק = חיובי   |   אדום = שלילי   |   אפור = ניטרלי'),
    fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('genizah_sentiment.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 הגרף נשמר: genizah_sentiment.png')

In [ ]:
# ============================================================
# גניזה שלב ג: מידול נושאים (LDA)
# ============================================================
from gensim import corpora
from gensim.models import LdaModel

GENIZAH_STOPWORDS = {
    'של', 'עם', 'את', 'אל', 'על', 'כי', 'לא', 'הוא', 'היא', 'הם',
    'זה', 'זו', 'כל', 'יש', 'אין', 'גם', 'אם', 'כן', 'לו', 'לה',
    'בן', 'בת', 'כך', 'בין', 'ביום', 'בחודש', 'בעיר', 'ממנו', 'אחר',
}

processed_lda_g = [
    [w for w in tokenize_hebrew(t)
     if w not in GENIZAH_STOPWORDS and len(w) >= 3]
    for t in list(GENIZAH_TEXTS.values())
]
dict_g  = corpora.Dictionary(processed_lda_g)
bow_g   = [dict_g.doc2bow(doc) for doc in processed_lda_g]

lda_g = LdaModel(
    corpus=bow_g, id2word=dict_g,
    num_topics=3, random_state=42, passes=40, alpha='auto')

TOPIC_LABELS = [rtl('דתי / ליטורגי'), rtl('מסחרי / משפטי'), rtl('קהילתי / אישי')]
colors_lda   = ['#8e44ad', '#e67e22', '#27ae60']

fig, axes = plt.subplots(1, 3, figsize=(15, 6))
for tid in range(3):
    topic       = lda_g.show_topic(tid, topn=8)
    words, wts  = zip(*topic)
    ax = axes[tid]
    ax.barh(range(len(words)), wts, color=colors_lda[tid], alpha=0.85)
    ax.set_yticks(range(len(words)))
    ax.set_yticklabels([rtl(w) for w in words], fontsize=11)
    ax.invert_yaxis()
    ax.set_title(TOPIC_LABELS[tid], fontsize=12, fontweight='bold',
                 color=colors_lda[tid])
    ax.set_xlabel(rtl('משקל'), fontsize=10)

plt.suptitle(rtl('מידול נושאים — LDA על קטעי הגניזה (3 נושאים)'),
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('genizah_lda.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 הגרף נשמר: genizah_lda.png')

In [ ]:
# ============================================================
# גניזה שלב ד: דמיון וקטורי בין קטעים
# ============================================================
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

doc_names_short = [k.split('(')[1].rstrip(')') for k in GENIZAH_TEXTS]
texts_list_g    = list(GENIZAH_TEXTS.values())

processed_g  = [tokenize_hebrew(t) for t in texts_list_g]
tfidf_g      = TfidfVectorizer(analyzer=lambda x: x)
tfidf_mat_g  = tfidf_g.fit_transform(processed_g)
sim_g        = cosine_similarity(tfidf_mat_g)

fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(sim_g, cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label=rtl('דמיון קוסינוס'))

tick_lbls = [rtl(n) for n in doc_names_short]
ax.set_xticks(range(len(tick_lbls)))
ax.set_xticklabels(tick_lbls, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(len(tick_lbls)))
ax.set_yticklabels(tick_lbls, fontsize=9)
ax.set_title(
    rtl('דמיון בין קטעי הגניזה — TF-IDF Cosine Similarity') + '\n' +
    rtl('צהוב כהה = קטעים דומים; לבן = קטעים שונים'),
    fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('genizah_similarity.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 הגרף נשמר: genizah_similarity.png')
print('\n📌 מה ניתן לראות:')
print('  קטעים מאותו סוג (מסחריים, דתיים, אישיים) צפויים להיות דומים יותר זה לזה')

## 📝 תרגילים

### תרגיל 1 – בסיסי ⭐

הריצו LDA עם מספר נושאים שונה (2, 4, 6, 8) ובדקו:  
אילו נושאים מתאימים יותר לנתונים שלנו?

### תרגיל 2 – בינוני ⭐⭐

הוסיפו ישויות חדשות ל-Gazetteer (מקומות, דמויות, ארגונים שאתם מכירים מהתחום).  
הריצו NER על טקסטים מהקורפוס שנאסף במחברת 1.

### תרגיל 3 – מתקדם ⭐⭐⭐

השתמשו ב-**AlephBERT** לניתוח רגש מתקדם:

```python
from transformers import pipeline
# nlp_he = pipeline("text-classification", model="avichr/heBERT_sentiment_analysis")
```

---

## 🔗 משאבים

- [AlephBERT](https://huggingface.co/onlplab/alephbert-base) – BERT לעברית
- [Gensim LDA Tutorial](https://radimrehurek.com/gensim/auto_examples/tutorials/run_lda.html)
- [pyLDAvis](https://github.com/bmabey/pyLDAvis) – ויזואליזציה של LDA
- Blei et al. (2003). *Latent Dirichlet Allocation*. JMLR.
- [Programming Historian: NLP](https://programminghistorian.org/)

---

## 🏠 משימת הבית

### מה להגיש:
1. **הריצו LDA על קורפוס אחר** – שנו את רשימת הדפים ב-`WIKI_PAGES` לנושא שמעניין אתכם (לפחות 15 דפים)
2. **תארו נושא אחד** – בחרו נושא אחד מהתוצאות וכתבו פסקה: על מה לדעתכם הנושא הזה? אילו מילות מפתח מרמזות על כך?
3. **הרהור ביקורתי** – ב-2–3 משפטים: מה המגבלות של שיטת LDA? מה היא לא יכולה לאתר?

### כיצד להגיש:
1. **File → Save a copy in Drive** – ודאו שהמחברת שמורה
2. **File → Download → Download .ipynb** – הורידו את הקובץ
3. העלו ל-Moodle

---

### 🤖 עצה: אם נתקעתם – שאלו את Gemini!
פתחו טאב חדש ב-[Google AI Studio](https://aistudio.google.com/) ושאלו:

```
אני לומד/ת NLP ומידול נושאים ב-Python.
נתקלתי בשגיאה הזו: "[הדביקו את השגיאה כאן]"
מה המשמעות ואיך אפשר לפתור?
```
